In [1]:

%matplotlib inline

Knowledge Distillation Tutorial
===============================

**Author**: [Alexandros Chariton](https://github.com/AlexandrosChrtn)


知识蒸馏是一种能够将知识从庞大、计算成本高的模型转移到较小模型的技术，且不会损失有效性。这使得模型能够在性能较弱的硬件上部署，从而加快评估速度并提升效率。

在本教程中，我们将开展一系列实验，专注于通过使用一个更强大的网络作为教师，来提高轻量级神经网络的准确性。轻量级网络的计算成本和速度将保持不变，我们的干预仅聚焦于其权重，而非其前向传播过程。该技术的应用场景包括无人机或移动电话等设备。在本教程中，我们不会使用任何外部软件包，因为所需的一切在 torch 和 torchvision 中均已提供。

在本教程中，您将学习：

如何修改模型类以提取隐藏表示并将其用于进一步计算

如何修改 PyTorch 中的常规训练循环，以在分类的交叉熵等损失基础上增加额外损失

如何通过使用更复杂的模型作为教师，提升轻量级模型的性能

前置要求
=============
1 个 GPU，4GB 内存

PyTorch v2.0 或更高版本

CIFAR-10 数据集（由脚本下载并保存在名为 /data 的目录中）

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# 如果有 GPU，会用 GPU 快速训练。如果没有，也能用 CPU 完成（虽然慢一些），但不会因为设备问题而无法运行
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


加载 CIFAR-10
================

CIFAR-10 是一个流行的图像数据集，包含十个类别。我们的目标是为每个输入图像预测以下类别之一。

![Example of CIFAR-10
images](https://pytorch.org/tutorials//../_static/img/cifar10.png)

输入图像： RGB 图像，它们有 3 个通道，大小为 32×32 像素。基本上，每张图像由 3 × 32 × 32 = 3072 个取值范围在 0 到 255 之间的数字来描述。神经网络中的一个常见做法是对输入进行归一化，这有多方面原因，包括避免常用激活函数出现饱和以及提高数值稳定性。
我们的归一化过程包括按每个通道减去均值并除以标准差。张量 mean=[0.485, 0.456, 0.406] 和 std=[0.229, 0.224, 0.225] 是预先计算好的，它们表示 CIFAR-10 中预定义的训练集子集里每个通道的均值和标准差。
注意，我们在测试集上也使用了这些数值，而不是重新从头计算均值和标准差。这是因为网络是在通过上述数值进行减除操作后生成的特征上训练的，我们希望保持一致性。此外，在实际应用中，我们无法计算测试集的均值和标准差，因为按照我们的假设，此时是无法访问这些数据的。
最后需要说明的是，我们通常将这个留出的数据集称为验证集，并在验证集上优化模型性能后，再使用另一个称为测试集的独立数据集进行评估。这样做是为了避免基于单一指标的贪婪且有偏的优化来选取模型。


In [4]:
# 下面我们正在对 CIFAR-10 数据集进行预处理。我们使用了一个随意选择的批大小 128。
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 加载CIFAR-10数据集:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

100%|██████████| 170M/170M [00:01<00:00, 93.2MB/s] 


<div style="background-color: #54c7ec; color: #fff; font-weight: 700; padding-left: 10px; padding-top: 5px; padding-bottom: 5px"><strong>NOTE:</strong></div>

<div style="background-color: #f3f4f7; padding-left: 10px; padding-top: 10px; padding-bottom: 10px; padding-right: 10px">

<p>本节仅适用于 CPU 用户，且只关心快速得到结果。仅在您希望进行小规模实验时才使用此选项。请记住，使用任何 GPU 运行代码都应该相当快。请只从训练/测试数据集中选择前 num_images_to_keep 张图像。 <code>num_images_to_keep</code> 图像用来训练/测试<pre><code>#from torch.utils.data import Subset</p>
<h1>num_images_to_keep = 2000</h1>
<h1>train_dataset = Subset(train_dataset, range(min(num_images_to_keep, 50_000)))</h1>
<h1>test_dataset = Subset(test_dataset, range(min(num_images_to_keep, 10_000)))</code></pre></h1>
```

</div>



数据加载器使用 batch_size=128，训练集打乱顺序，测试集不打乱。num_workers=2 利用多核并行加载，在 Kaggle/Colab 上运行稳定。若本地显存不足，可将 batch_size 减半并相应调整学习率。

In [6]:
#图像加载
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

**定义模型类与辅助函数**

接下来，我们需要定义模型类。这里需要设置几个用户自定义参数。我们使用两种不同的架构，并在实验中保持滤波器数量固定，以确保公平比较。两种架构均为卷积神经网络（CNN），包含不同数量的卷积层作为特征提取器，后接一个包含10个类别的分类器。学生模型的滤波器数量和神经元数量更少。

## 模型结构对比

知识蒸馏的核心是使用一个**容量更大的教师模型**指导**轻量级学生模型**。本实验中，教师与学生均为自定义 CNN，具体结构对比如下：

| 模型       | 卷积层数 | 最大池化次数 | 全连接层数 | 参数量（估算） | 设计定位     |
|------------|----------|--------------|------------|----------------|--------------|
| **DeepNN**（教师） | 5        | 2            | 2（含 Dropout） | ~ 2.1M         | 高容量，提取丰富特征 |
| **LightNN**（学生） | 3        | 2            | 2（含 Dropout） | ~ 0.5M         | 轻量级，适合部署 |

教师模型通过更深的卷积层和更多通道数捕获复杂模式，为学生提供高质量的软标签；学生模型参数量仅为教师的约 1/4，通过蒸馏学习教师输出的类间相似性，从而弥补自身容量不足。

In [7]:
# 用作教师的深层神经网络类：
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# 用作学生的轻量级神经网络类：
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

我们使用两个函数来帮助生成和评估原始分类任务的结果。其中一个函数名为 train，它接受以下参数：

model：要通过该函数进行训练（更新其权重）的模型实例。

train_loader：我们在前面定义的 train_loader，其作用是将数据馈送到模型中。

epochs：我们在数据集上循环遍历的次数。

learning_rate：学习率决定了我们向收敛迈进的步长大小。步长过大或过小都可能不利。

device：决定运行任务所用的设备，可以是 CPU 或 GPU，取决于可用性。

我们的测试函数与之类似，但它将使用 test_loader 来从测试集中加载图像。

![Train both networks with Cross-Entropy. The student will be used as a
baseline:](https://pytorch.org/tutorials//../_static/img/knowledge_distillation/ce_only.png){.align-center}


In [9]:
def train(model, train_loader, epochs, learning_rate, device):
    # 定义损失函数：交叉熵，适用于多分类任务
    criterion = nn.CrossEntropyLoss()
    # 定义优化器：Adam，使用给定的学习率
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # 将模型设置为训练模式（启用 dropout、batch norm 等）
    model.train()

    # 开始训练循环
    for epoch in range(epochs):
        # 记录当前 epoch 的累计损失
        running_loss = 0.0
        # 遍历训练数据加载器，每次返回一个批次
        for inputs, labels in train_loader:
            # 将输入图像和标签移动到指定设备（GPU/CPU）
            inputs, labels = inputs.to(device), labels.to(device)

            # 梯度清零，避免累积
            optimizer.zero_grad()
            # 前向传播，得到模型输出（logits）
            outputs = model(inputs)

            # 计算当前批次的损失
            loss = criterion(outputs, labels)
            # 反向传播，计算梯度
            loss.backward()
            # 更新模型参数
            optimizer.step()

            # 累加当前批次的损失值（用于打印）
            running_loss += loss.item()

        # 打印当前 epoch 的平均损失
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


def test(model, test_loader, device):
    # 将模型移动到指定设备
    model.to(device)
    # 设置为评估模式（关闭 dropout、batch norm 使用全局统计量）
    model.eval()

    # 记录正确预测的数量
    correct = 0
    # 记录总样本数
    total = 0

    # 禁用梯度计算（减少内存消耗，加速推理）
    with torch.no_grad():
        # 遍历测试数据加载器
        for inputs, labels in test_loader:
            # 将数据移动到指定设备
            inputs, labels = inputs.to(device), labels.to(device)

            # 前向传播
            outputs = model(inputs)
            # 获取每个样本预测的类别（最大 logit 对应的索引）
            _, predicted = torch.max(outputs.data, 1)

            # 更新总样本数
            total += labels.size(0)
            # 累加正确预测的样本数
            correct += (predicted == labels).sum().item()

    # 计算准确率（百分比）
    accuracy = 100 * correct / total
    # 打印结果
    print(f"Test Accuracy: {accuracy:.2f}%")
    # 返回准确率，供后续分析使用
    return accuracy

**交叉熵训练**

为了结果的可复现性，我们需要设置 torch 的随机种子。我们会使用不同的方法训练网络，为了公平比较，用相同的权重初始化网络是合理的。首先，使用交叉熵训练教师网络：



In [10]:
# 设置随机种子，保证结果可复现
torch.manual_seed(42)

# 创建教师模型 DeepNN 实例，指定类别数为 10（CIFAR-10），并移动到指定设备（GPU/CPU）
nn_deep = DeepNN(num_classes=10).to(device)

# 调用训练函数，对教师模型进行训练：训练数据加载器 train_loader，训练 10 个 epoch，学习率 0.001，设备为 device
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)

# 测试教师模型在测试集上的准确率，并将结果保存到变量 test_accuracy_deep
test_accuracy_deep = test(nn_deep, test_loader, device)

# 实例化轻量级网络（学生模型）的注释
# 再次设置随机种子，保证学生模型的初始化与教师模型初始化时的随机状态一致（方便对比）
torch.manual_seed(42)

# 创建学生模型 LightNN 实例，指定类别数为 10，并移动到指定设备
nn_light = LightNN(num_classes=10).to(device)

Epoch 1/10, Loss: 1.32787046362372
Epoch 2/10, Loss: 0.8685831592211029
Epoch 3/10, Loss: 0.6810062911809253
Epoch 4/10, Loss: 0.5393115282820924
Epoch 5/10, Loss: 0.42170425597816474
Epoch 6/10, Loss: 0.3111089274020451
Epoch 7/10, Loss: 0.22001730290520222
Epoch 8/10, Loss: 0.178567489261365
Epoch 9/10, Loss: 0.14478197818636285
Epoch 10/10, Loss: 0.11751037388754165
Test Accuracy: 75.12%


我们再实例化一个轻量级网络模型，以便比较它们的性能。反向传播对权重初始化非常敏感，因此我们需要确保这两个网络具有完全相同的初始化。

In [11]:
# 设置随机种子，保证后续操作（如模型参数初始化）结果可复现
torch.manual_seed(42)

# 创建学生模型 LightNN 的新实例，指定类别数为 10（CIFAR-10），并移动到指定设备（GPU/CPU）
new_nn_light = LightNN(num_classes=10).to(device)

为了保证实验的可复现性，我们在创建学生模型前固定了随机种子 42。为了快速确认两次实例化的初始状态一致，我们打印了第一个卷积层权重的 L2 范数，两次输出相同（均为 2.32736），这暗示了权重初始化完全相同。如果需要更严格的验证，可以使用 torch.allclose 比较两个模型的权重张量，差值应为零。

In [13]:
# 打印初始学生模型 nn_light 的第一个卷积层（features[0]）的权重矩阵的范数（默认为 L2 范数），并输出数值
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())

# 打印新创建的学生模型 new_nn_light 的第一个卷积层（features[0]）的权重矩阵的范数，同样输出数值
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())

Norm of 1st layer of nn_light: 2.327361822128296
Norm of 1st layer of new_nn_light: 2.327361822128296


要打印每个模型的总参数量：

In [10]:
total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN parameters: {total_params_deep}")
total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN parameters: {total_params_light}")

DeepNN parameters: 1,186,986
LightNN parameters: 267,738


训练轻量网络（学生）单独使用交叉熵损失：


In [14]:
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.470008578751703
Epoch 2/10, Loss: 1.16062329919137
Epoch 3/10, Loss: 1.0290574022876027
Epoch 4/10, Loss: 0.9197332506899334
Epoch 5/10, Loss: 0.8402177822559386
Epoch 6/10, Loss: 0.7738902613025187
Epoch 7/10, Loss: 0.7064248945402063
Epoch 8/10, Loss: 0.6493706238239317
Epoch 9/10, Loss: 0.5993007402438337
Epoch 10/10, Loss: 0.5488478273839292
Test Accuracy: 70.48%


如我们所见，基于测试准确率，我们现在可以将用作教师的更深层网络与作为学生的轻量级网络进行比较。到目前为止，我们的学生尚未与教师进行任何干预，因此这一性能是学生自身达到的。

In [12]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light_ce:.2f}%")

Teacher accuracy: 74.84%
Student accuracy: 70.55%


现在，让我们尝试通过引入教师网络来提升学生网络的测试准确率。知识蒸馏是实现这一目标的一种直接技术，其基础在于两个网络都输出关于各类别的概率分布。因此，这两个网络共享相同数量的输出神经元。该方法通过在传统交叉熵损失之外增加一个额外损失来实现，该额外损失基于教师网络的 softmax 输出。其假设是：一个经过良好训练的教师网络的输出激活携带有额外信息，学生网络在训练过程中可以利用这些信息。原始工作表明，利用软目标中较小概率的比值，有助于实现深度神经网络的基本目标，即构建一个数据上的相似性结构，使相似对象在映射空间中更接近。例如，在 CIFAR-10 中，一辆卡车如果带有轮子，可能会被误认为汽车或飞机，但不太可能被误认为狗。因此，我们有理由认为，有价值的信息不仅存在于一个训练良好的模型的最高预测中，也存在于整个输出分布中。然而，单独的交叉熵不足以充分利用这些信息，因为非预测类别的激活值往往非常小，使得传播的梯度无法有意义地改变权重来构建理想的向量空间。

在我们继续定义第一个引入师生动态的辅助函数时，需要加入几个额外参数：

T：温度，用于控制输出分布的平滑度。较大的 T 会使分布更平滑，从而较小的概率获得更大的提升。

soft_target_loss_weight：分配给我们将要引入的额外目标的权重。

ce_loss_weight：分配给交叉熵的权重。调整这些权重会使网络倾向于优化其中某个目标。

![Distillation loss is calculated from the logits of the networks. It
only returns gradients to the
student:](https://pytorch.org/tutorials//../_static/img/knowledge_distillation/distillation_output_loss.png){.align-center}


In [15]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    # 定义硬标签损失：交叉熵损失
    ce_loss = nn.CrossEntropyLoss()
    # 定义优化器：Adam，优化学生模型参数
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    # 将教师模型设置为评估模式（关闭 dropout 和 batch norm 的训练行为，且不计算梯度）
    teacher.eval()  # Teacher set to evaluation mode
    # 将学生模型设置为训练模式（启用 dropout 等）
    student.train() # Student to train mode

    # 开始训练循环，共 epochs 轮
    for epoch in range(epochs):
        running_loss = 0.0
        # 遍历训练数据加载器，每次返回一个批次
        for inputs, labels in train_loader:
            # 将输入图像和标签移动到指定设备（GPU/CPU）
            inputs, labels = inputs.to(device), labels.to(device)

            # 梯度清零，避免累积
            optimizer.zero_grad()

            # 教师模型前向传播，但不计算梯度（因为教师参数固定）
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # 学生模型前向传播，需要计算梯度以便更新参数
            student_logits = student(inputs)

            # 软化教师 logits：应用 softmax 并除以温度 T，得到软标签分布
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            # 软化学生 logits：应用 log_softmax 并除以温度 T，用于计算 KL 散度
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)

            # 计算软标签损失（KL 散度的一种实现），乘以 T^2 以补偿梯度尺度（原论文建议）
            # soft_targets * (soft_targets.log() - soft_prob) 是每个样本的 KL 散度分量
            # 除以 soft_prob.size()[0] 得到批次平均，再乘以 T^2
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)

            # 计算硬标签损失（交叉熵），使用学生原始 logits（未除以温度）
            label_loss = ce_loss(student_logits, labels)

            # 加权组合软损失和硬损失，权重由用户指定
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            # 反向传播，计算梯度
            loss.backward()
            # 更新学生模型参数
            optimizer.step()

            # 累加当前批次的损失值
            running_loss += loss.item()

        # 打印当前 epoch 的平均损失
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

# 调用蒸馏训练函数，参数：
# teacher: 已训练好的教师模型 nn_deep
# student: 新实例化的学生模型 new_nn_light（保证初始状态与基线一致）
# train_loader: 训练数据加载器
# epochs=10: 训练 10 个 epoch
# learning_rate=0.001: Adam 学习率
# T=2: 温度参数，控制软标签平滑程度
# soft_target_loss_weight=0.25: 软标签损失权重
# ce_loss_weight=0.75: 硬标签损失权重
# device: 设备（GPU/CPU）
train_knowledge_distillation(teacher=nn_deep, student=new_nn_light, train_loader=train_loader, epochs=10, learning_rate=0.001, T=2, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)

# 测试蒸馏后的学生模型在测试集上的准确率
test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

# 打印对比结果：教师准确率、学生单独训练准确率、学生蒸馏后准确率
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Epoch 1/10, Loss: 2.4106471206221127
Epoch 2/10, Loss: 1.893745611085916
Epoch 3/10, Loss: 1.6689254374760192
Epoch 4/10, Loss: 1.5059961946419134
Epoch 5/10, Loss: 1.3782550203220925
Epoch 6/10, Loss: 1.2615812859876687
Epoch 7/10, Loss: 1.1683015120608726
Epoch 8/10, Loss: 1.0777355610866985
Epoch 9/10, Loss: 1.0024140548828009
Epoch 10/10, Loss: 0.9346689331866896
Test Accuracy: 70.52%
Teacher accuracy: 75.12%
Student accuracy without teacher: 70.48%
Student accuracy with CE + KD: 70.52%


余弦损失最小化运行
============================

您可以自由调整控制 softmax 函数柔软度的温度参数以及损失系数。在神经网络中，很容易在主要目标之外添加额外的损失函数，以实现更好的泛化等目标。让我们尝试为学生网络加入一个目标，但现在我们关注的是它们的隐藏状态而非输出层。我们的目标是，通过引入一个朴素的损失函数，将教师网络表示的信息传递给学生。该损失函数的最小化意味着，随着损失下降，随后传入分类器的扁平化向量变得更加相似。当然，教师网络不会更新其权重，因此最小化仅依赖于学生网络的权重。这种方法背后的逻辑是，我们假设教师模型具有更好的内部表示，而学生在没有外部干预的情况下不太可能达到这种表示，因此我们人为地推动学生模仿教师网络的内部表示。然而，这是否最终能帮助学生并不直接明了，因为推动轻量级网络达到这一点可能是好事（假设我们找到了一个能带来更好测试准确率的内部表示），但也可能有害，因为网络具有不同的架构，学生不具备与教师相同的学习能力。换句话说，这两个向量（学生的和教师的）没有理由在分量上逐一匹配。学生可能达到一种与教师表示等价的排列，其效率是一样的。尽管如此，我们仍然可以运行一个快速实验来了解这种方法的影响。我们将使用 CosineEmbeddingLoss，其公式如下：



![Formula for
CosineEmbeddingLoss](https://pytorch.org/tutorials//../_static/img/knowledge_distillation/cosine_embedding_loss.png){.align-center
width="450px"}

显然，我们首先需要解决一个问题。当我们将蒸馏应用于输出层时，我们提到两个网络具有相同数量的神经元，等于类别数。然而，对于卷积层之后的层，情况并非如此。这里，在最后一个卷积层展平后，教师网络的神经元数量多于学生网络。我们的损失函数接受两个等维度的向量作为输入，因此我们需要以某种方式使它们匹配。我们将通过在教师网络的卷积层之后添加一个平均池化层来解决这个问题，以降低其维度，使其与学生网络匹配。

为此，我们将修改模型类，或者创建新的模型类。现在，前向函数不仅返回网络的 logits，还返回卷积层之后的展平隐藏表示。我们在修改后的教师网络中引入上述池化操作。




In [17]:
# 定义一个修改后的深层网络类，用于返回特征向量（用于余弦相似度损失）
class ModifiedDeepNNCosine(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedDeepNNCosine, self).__init__()
        # 特征提取部分，结构与原 DeepNN 完全一致
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 分类器部分，也与原 DeepNN 一致
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # 前向传播：通过特征提取器
        x = self.features(x)
        # 展平特征图，得到一维特征向量（用于分类）
        flattened_conv_output = torch.flatten(x, 1)
        # 通过分类器得到 logits
        x = self.classifier(flattened_conv_output)
        # 对展平后的特征向量进行平均池化（降维），用于余弦相似度损失
        flattened_conv_output_after_pooling = torch.nn.functional.avg_pool1d(flattened_conv_output, 2)
        # 返回 logits 和池化后的特征向量
        return x, flattened_conv_output_after_pooling

# 创建一个类似的轻量级学生类，也返回特征向量（不做池化）
class ModifiedLightNNCosine(nn.Module):
    def __init__(self, num_classes=10):
        super(ModifiedLightNNCosine, self).__init__()
        # 特征提取部分，与原 LightNN 一致
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 分类器部分，与原 LightNN 一致
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # 前向传播：特征提取
        x = self.features(x)
        # 展平特征图，得到一维特征向量
        flattened_conv_output = torch.flatten(x, 1)
        # 通过分类器得到 logits
        x = self.classifier(flattened_conv_output)
        # 返回 logits 和原始特征向量（未池化）
        return x, flattened_conv_output

# 实例化修改后的深层网络（教师），并加载之前训练好的教师模型权重
modified_nn_deep = ModifiedDeepNNCosine(num_classes=10).to(device)
modified_nn_deep.load_state_dict(nn_deep.state_dict())

# 验证修改后的教师模型与原教师模型第一层权重范数是否相同（应相同，因为权重相同）
print("Norm of 1st layer for deep_nn:", torch.norm(nn_deep.features[0].weight).item())
print("Norm of 1st layer for modified_deep_nn:", torch.norm(modified_nn_deep.features[0].weight).item())

# 初始化一个修改后的轻量级网络（学生），使用相同的随机种子，以便与之前的实验保持初始化一致
torch.manual_seed(42)
modified_nn_light = ModifiedLightNNCosine(num_classes=10).to(device)
# 打印第一层权重的范数，用于记录初始状态
print("Norm of 1st layer:", torch.norm(modified_nn_light.features[0].weight).item())

Norm of 1st layer for deep_nn: 7.5072197914123535
Norm of 1st layer for modified_deep_nn: 7.5072197914123535
Norm of 1st layer: 2.327361822128296


当然，我们需要修改训练循环，因为现在模型返回一个元组 (logits, hidden_representation)。使用一个样本输入张量，我们可以打印它们的形状。


In [18]:
# 创建一个样本输入张量，模拟一个批次的 CIFAR-10 图像
# 形状含义：128（批次大小）、3（RGB通道）、32（高度）、32（宽度）
sample_input = torch.randn(128, 3, 32, 32).to(device) # 将张量移动到指定设备（GPU/CPU）

# 将样本输入传入修改后的学生模型（ModifiedLightNNCosine）
# 模型返回一个元组：(logits, hidden_representation)
logits, hidden_representation = modified_nn_light(sample_input)

# 打印学生模型输出的 logits 形状（批次大小 × 类别数）
print("Student logits shape:", logits.shape) # 预期形状：torch.Size([128, 10])

# 打印学生模型输出的隐藏表示形状（批次大小 × 隐藏特征维度）
print("Student hidden representation shape:", hidden_representation.shape) # 预期形状：torch.Size([128, 1024])

# 将同样的样本输入传入修改后的教师模型（ModifiedDeepNNCosine）
# 教师模型也返回元组 (logits, hidden_representation)
logits, hidden_representation = modified_nn_deep(sample_input)

# 打印教师模型输出的 logits 形状（同样为 128 × 10）
print("Teacher logits shape:", logits.shape) # 形状：[128, 10]

# 打印教师模型输出的隐藏表示形状
# 由于教师模型中添加了平均池化，其隐藏维度被降低到与学生一致（1024）
print("Teacher hidden representation shape:", hidden_representation.shape) # 预期形状：torch.Size([128, 1024])

Student logits shape: torch.Size([128, 10])
Student hidden representation shape: torch.Size([128, 1024])
Teacher logits shape: torch.Size([128, 10])
Teacher hidden representation shape: torch.Size([128, 1024])


在我们的案例中，hidden_representation_size 为 1024。这是学生模型最后一个卷积层展平后的特征图，如您所见，它是其分类器的输入。教师模型同样是 1024，因为我们通过 avg_pool1d 将原本的 2048 降到了这个维度。此处应用的损失仅影响学生模型在损失计算之前的权重，换句话说，它不会影响学生模型的分类器。修改后的训练循环如下：

![In Cosine Loss minimization, we want to maximize the cosine similarity
of the two representations by returning gradients to the
student:](https://pytorch.org/tutorials//../_static/img/knowledge_distillation/cosine_loss_distillation.png){.align-center}


In [16]:
def train_cosine_loss(teacher, student, train_loader, epochs, learning_rate, hidden_rep_loss_weight, ce_loss_weight, device):
    """
    使用余弦嵌入损失（CosineEmbeddingLoss）训练学生模型，使其隐藏表示接近教师模型的隐藏表示。

    参数：
        teacher: 教师模型（已训练好，处于评估模式）
        student: 学生模型（待训练）
        train_loader: 训练数据加载器
        epochs: 训练轮数
        learning_rate: 学习率
        hidden_rep_loss_weight: 余弦损失的权重
        ce_loss_weight: 交叉熵损失的权重
        device: 运行设备（cuda/cpu）
    """

    # 定义交叉熵损失（用于分类）
    ce_loss = nn.CrossEntropyLoss()
    # 定义余弦嵌入损失（用于衡量两个向量之间的相似度）
    cosine_loss = nn.CosineEmbeddingLoss()
    # 使用 Adam 优化器更新学生模型的参数
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    # 将教师模型和学生模型移动到指定设备
    teacher.to(device)
    student.to(device)
    # 教师模型设置为评估模式（不更新权重，不启用 dropout/batch norm 的训练行为）
    teacher.eval()
    # 学生模型设置为训练模式（启用 dropout 等）
    student.train()

    # 开始逐轮训练
    for epoch in range(epochs):
        running_loss = 0.0  # 记录当前 epoch 的总损失

        # 遍历训练数据加载器
        for inputs, labels in train_loader:
            # 将输入数据和标签移动到指定设备
            inputs, labels = inputs.to(device), labels.to(device)

            # 清零优化器的梯度
            optimizer.zero_grad()

            # ---------- 教师模型前向传播（不计算梯度） ----------
            with torch.no_grad():
                # 教师模型返回 (logits, hidden_representation)，我们只需要隐藏表示
                _, teacher_hidden_representation = teacher(inputs)

            # ---------- 学生模型前向传播 ----------
            # 学生模型返回 (logits, hidden_representation)
            student_logits, student_hidden_representation = student(inputs)

            # ---------- 计算余弦损失 ----------
            # 目标是让 student_hidden_representation 和 teacher_hidden_representation 的方向尽可能一致
            # CosineEmbeddingLoss 的 target=1 表示希望两个向量相似（余弦相似度高）
            # 这里为批次中的每个样本创建一个值为 1 的目标张量
            target = torch.ones(inputs.size(0)).to(device)
            hidden_rep_loss = cosine_loss(student_hidden_representation, teacher_hidden_representation, target)

            # ---------- 计算分类损失（交叉熵） ----------
            label_loss = ce_loss(student_logits, labels)

            # ---------- 总损失 = 权重1 * 余弦损失 + 权重2 * 交叉熵损失 ----------
            loss = hidden_rep_loss_weight * hidden_rep_loss + ce_loss_weight * label_loss

            # 反向传播，计算梯度
            loss.backward()
            # 更新学生模型的权重
            optimizer.step()

            # 累加当前 batch 的损失值
            running_loss += loss.item()

        # 每个 epoch 结束后打印平均损失
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

出于同样的原因，我们需要修改测试函数。在这里，我们忽略模型返回的隐藏表示。

In [17]:
def test_multiple_outputs(model, test_loader, device):
    """
    测试返回元组 (logits, hidden_representation) 的模型。
    参数：
        model: 待测试的模型（例如 ModifiedLightNNCosine 或 ModifiedDeepNNCosine）
        test_loader: 测试数据加载器
        device: 运行设备（cuda/cpu）
    """
    # 将模型移动到指定设备
    model.to(device)
    # 设置模型为评估模式（禁用 dropout 等）
    model.eval()

    correct = 0   # 记录正确预测的数量
    total = 0     # 记录总样本数量

    # 禁用梯度计算，减少内存占用并加速
    with torch.no_grad():
        # 遍历测试数据加载器
        for inputs, labels in test_loader:
            # 将输入数据和标签移动到指定设备
            inputs, labels = inputs.to(device), labels.to(device)

            # 前向传播：模型返回 (logits, hidden_representation)
            # 我们只需要 logits，因此用 _ 忽略第二个返回值
            outputs, _ = model(inputs)

            # 获取每个样本预测的类别（得分最高的索引）
            _, predicted = torch.max(outputs.data, 1)

            # 更新总样本数（当前批次大小）
            total += labels.size(0)
            # 累计正确预测数
            correct += (predicted == labels).sum().item()

    # 计算准确率百分比
    accuracy = 100 * correct / total
    # 打印准确率
    print(f"Test Accuracy: {accuracy:.2f}%")
    # 返回准确率，以便后续分析
    return accuracy

在这种情况下，我们可以轻松地将知识蒸馏和余弦损失最小化整合到同一个函数中。在师生范式中，结合多种方法以取得更优性能是常见做法。目前，我们可以先运行一个简单的训练-测试流程。

In [18]:
# 调用 train_cosine_loss 函数，使用余弦嵌入损失训练学生模型，使其隐藏表示向教师模型对齐
# teacher: 修改后的教师模型（ModifiedDeepNNCosine），已加载预训练权重，作为目标
# student: 修改后的学生模型（ModifiedLightNNCosine），将从随机初始化开始训练
# train_loader: 训练数据加载器（CIFAR-10 训练集）
# epochs: 训练轮数，这里设为 10
# learning_rate: 学习率 0.001
# hidden_rep_loss_weight: 余弦损失的权重，设为 0.25
# ce_loss_weight: 交叉熵损失的权重，设为 0.75（总损失中分类任务占主导）
# device: 运行设备（如 cuda）
train_cosine_loss(teacher=modified_nn_deep, student=modified_nn_light, train_loader=train_loader, epochs=10, learning_rate=0.001, hidden_rep_loss_weight=0.25, ce_loss_weight=0.75, device=device)

# 调用 test_multiple_outputs 函数，在测试集上评估训练后的学生模型性能
# modified_nn_light: 刚刚完成训练的学生模型（现在已包含余弦损失训练后的权重）
# test_loader: 测试数据加载器（CIFAR-10 测试集）
# device: 运行设备
# 该函数返回测试准确率，并打印出来，同时将准确率保存在变量 test_accuracy_light_ce_and_cosine_loss 中
test_accuracy_light_ce_and_cosine_loss = test_multiple_outputs(modified_nn_light, test_loader, device)

Epoch 1/10, Loss: 1.303395491731746
Epoch 2/10, Loss: 1.0718137272788435
Epoch 3/10, Loss: 0.9716343000111982
Epoch 4/10, Loss: 0.8970773404516528
Epoch 5/10, Loss: 0.8408327575229928
Epoch 6/10, Loss: 0.7944081376885515
Epoch 7/10, Loss: 0.7561445079191261
Epoch 8/10, Loss: 0.7190640538244906
Epoch 9/10, Loss: 0.6802593225713276
Epoch 10/10, Loss: 0.6555877606887037
Test Accuracy: 70.83%


**中间回归器运行**

我们的朴素最小化方法并不能保证更好的结果，原因有多方面，其中之一是向量的维度。对于高维向量，余弦相似度通常比欧氏距离效果更好，但我们处理的是每个都有1024个分量的向量，因此从中提取有意义的相似度要困难得多。此外，正如我们所提到的，推动教师与学生的隐藏表示达到匹配并没有理论支持。我们并没有充分的理由去追求这些向量的一一匹配。我们将通过引入一个额外的网络（称为回归器）来提供最后一个训练干预的例子。其目标是：首先提取教师模型在某一卷积层之后的特征图，然后提取学生模型在某一卷积层之后的特征图，最后尝试匹配这些特征图。但这一次，我们将在网络之间引入一个回归器来辅助匹配过程。该回归器是可训练的，理想情况下会比我们朴素的余弦损失最小化方案做得更好。它的主要任务是匹配这些特征图的维度，以便我们能够在教师和学生之间正确定义一个损失函数。定义这样的损失函数提供了一条“教学路径”，本质上就是一个用于反向传播梯度的通路，从而改变学生模型的权重。聚焦于我们原始网络中每个分类器之前的卷积层的输出，其形状如下：

In [19]:
# 仅将样本输入通过卷积特征提取器部分（不经过分类器）
# 学生模型的 features 模块是一系列卷积层，输出特征图
convolutional_fe_output_student = nn_light.features(sample_input)

# 教师模型的 features 模块也是卷积层，输出特征图
convolutional_fe_output_teacher = nn_deep.features(sample_input)

# 打印形状，观察两个模型卷积层输出的尺寸差异
print("Student's feature extractor output shape: ", convolutional_fe_output_student.shape)
print("Teacher's feature extractor output shape: ", convolutional_fe_output_teacher.shape)

Student's feature extractor output shape:  torch.Size([128, 16, 8, 8])
Teacher's feature extractor output shape:  torch.Size([128, 32, 8, 8])


我们为教师设置了 32 个滤波器，为学生设置了 16 个滤波器。我们将引入一个可训练层，用于将学生的特征图转换为与教师特征图形状一致的形式。具体实现中，我们修改轻量级类，使其在通过一个中间回归器（该回归器负责匹配卷积特征图的尺寸）之后返回隐藏状态；同时修改教师类，使其直接返回最后一个卷积层的输出，而不进行池化或展平操作。

![The trainable layer matches the shapes of the intermediate tensors and
Mean Squared Error (MSE) is properly
defined:](https://pytorch.org/tutorials//../_static/img/knowledge_distillation/fitnets_knowledge_distill.png){.align-center}


In [20]:
class ModifiedDeepNNRegressor(nn.Module):
    """
    修改后的教师模型，返回 logits 和最后一个卷积层的特征图（不展平、不池化）。
    用于与学生的回归器输出进行匹配（特征图空间尺寸一致，通道数可能不同）。
    """
    def __init__(self, num_classes=10):
        super(ModifiedDeepNNRegressor, self).__init__()
        # 特征提取器：一系列卷积层，输出特征图
        self.features = nn.Sequential(
            # 输入 3 通道 -> 128 通道，3x3 卷积，保持空间尺寸
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            # 128 -> 64 通道
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # 2x2 最大池化，空间尺寸减半（32 -> 16）
            nn.MaxPool2d(kernel_size=2, stride=2),
            # 64 -> 64 通道
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            # 64 -> 32 通道
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            # 第二次池化，空间尺寸再减半（16 -> 8）
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        # 分类器：全连接层，输入为展平后的特征（32*8*8 = 2048）
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        # 通过特征提取器，输出形状 [batch, 32, 8, 8]
        x = self.features(x)
        # 保存卷积特征图（未展平），供后续损失计算使用
        conv_feature_map = x
        # 展平特征图，用于分类器
        x = torch.flatten(x, 1)          # [batch, 2048]
        # 通过分类器得到 logits
        x = self.classifier(x)           # [batch, num_classes]
        # 返回 (logits, 卷积特征图)
        return x, conv_feature_map


class ModifiedLightNNRegressor(nn.Module):
    """
    修改后的学生模型，包含一个额外的回归器（卷积层），
    用于将学生的特征图通道数提升到与教师一致（16 -> 32），
    从而可以在特征图层面计算损失（如 MSE 或余弦损失）。
    """
    def __init__(self, num_classes=10):
        super(ModifiedLightNNRegressor, self).__init__()
        # 特征提取器：轻量级卷积层，输出特征图 [batch, 16, 8, 8]
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),   # 32 -> 16
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),   # 16 -> 8
        )
        # 回归器：一个额外的卷积层，将学生特征图的通道数从 16 提升到 32，
        # 保持空间尺寸 8x8，从而与教师特征图的空间尺寸和通道数匹配
        self.regressor = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1)   # 输出 [batch, 32, 8, 8]
        )
        # 分类器：输入为展平后的原始特征（16*8*8 = 1024）
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # 通过特征提取器，得到 [batch, 16, 8, 8]
        x = self.features(x)
        # 将特征图传入回归器，得到与教师通道数一致的特征图 [batch, 32, 8, 8]
        regressor_output = self.regressor(x)
        # 同时保留原始特征图，展平后用于分类
        x_flat = torch.flatten(x, 1)      # [batch, 1024]
        # 通过分类器得到 logits
        logits = self.classifier(x_flat)  # [batch, num_classes]
        # 返回 (logits, 回归器输出的特征图) 用于与教师的特征图计算损失
        return logits, regressor_output

之后，我们需要再次更新训练循环。这一次，我们提取学生的回归器输出和教师的特征图，在这些张量上计算 MSE（它们形状完全相同，因此计算是合理的），并基于该损失进行反向传播，同时还要加上常规的分类任务交叉熵损失。


In [21]:
def train_mse_loss(teacher, student, train_loader, epochs, learning_rate, feature_map_weight, ce_loss_weight, device):
    """
    使用 MSE 损失训练学生模型，使其回归器输出的特征图与教师的特征图逐元素接近。
    同时保留交叉熵损失用于分类。
    """
    # 定义交叉熵损失（用于分类）
    ce_loss = nn.CrossEntropyLoss()
    # 定义均方误差损失（用于特征图匹配）
    mse_loss = nn.MSELoss()
    # 使用 Adam 优化器更新学生模型的参数
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    # 将教师模型和学生模型移动到指定设备
    teacher.to(device)
    student.to(device)
    # 教师模型设置为评估模式（不更新权重，不启用 dropout/batch norm 的训练行为）
    teacher.eval()
    # 学生模型设置为训练模式（启用 dropout 等）
    student.train()

    # 开始逐轮训练
    for epoch in range(epochs):
        running_loss = 0.0  # 记录当前 epoch 的总损失

        # 遍历训练数据加载器
        for inputs, labels in train_loader:
            # 将输入数据和标签移动到指定设备
            inputs, labels = inputs.to(device), labels.to(device)

            # 清零优化器的梯度
            optimizer.zero_grad()

            # ---------- 教师模型前向传播（不计算梯度） ----------
            with torch.no_grad():
                # 教师模型返回 (logits, feature_map)，这里只需要特征图
                _, teacher_feature_map = teacher(inputs)

            # ---------- 学生模型前向传播 ----------
            # 学生模型返回 (logits, regressor_feature_map)
            student_logits, regressor_feature_map = student(inputs)

            # ---------- 计算特征图匹配损失（MSE） ----------
            # regressor_feature_map 与 teacher_feature_map 形状相同（[batch, 32, 8, 8]）
            hidden_rep_loss = mse_loss(regressor_feature_map, teacher_feature_map)

            # ---------- 计算分类损失（交叉熵） ----------
            label_loss = ce_loss(student_logits, labels)

            # ---------- 总损失 = 权重1 * MSE 损失 + 权重2 * 交叉熵损失 ----------
            loss = feature_map_weight * hidden_rep_loss + ce_loss_weight * label_loss

            # 反向传播，计算梯度
            loss.backward()
            # 更新学生模型的权重
            optimizer.step()

            # 累加当前 batch 的损失值
            running_loss += loss.item()

        # 每个 epoch 结束后打印平均损失
        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

# 注意：测试函数与之前完全相同，因为我们只关心最终的分类准确率
# 测试函数 test_multiple_outputs 已经定义过，这里直接使用

# ---------- 初始化学生模型（带回归器） ----------
# 设置随机种子，保证可复现性
torch.manual_seed(42)
# 创建轻量级学生模型（带回归器），并移动到设备
modified_nn_light_reg = ModifiedLightNNRegressor(num_classes=10).to(device)

# ---------- 初始化教师模型（带特征图输出） ----------
# 创建修改后的教师模型（返回特征图），并移动到设备
modified_nn_deep_reg = ModifiedDeepNNRegressor(num_classes=10).to(device)
# 从已经训练好的原始教师模型（nn_deep）加载权重，保证教师能力不变
modified_nn_deep_reg.load_state_dict(nn_deep.state_dict())

# ---------- 使用 MSE 损失训练学生模型 ----------
train_mse_loss(teacher=modified_nn_deep_reg,       # 教师模型（提供特征图目标）
               student=modified_nn_light_reg,      # 学生模型（待训练）
               train_loader=train_loader,          # 训练数据加载器
               epochs=10,                          # 训练轮数
               learning_rate=0.001,                # 学习率
               feature_map_weight=0.25,            # 特征图损失的权重
               ce_loss_weight=0.75,                # 交叉熵损失的权重
               device=device)                      # 运行设备

# ---------- 在测试集上评估学生模型 ----------
# 使用之前定义的 test_multiple_outputs 函数，它忽略第二个返回值（特征图），只基于 logits 计算准确率
test_accuracy_light_ce_and_mse_loss = test_multiple_outputs(modified_nn_light_reg, test_loader, device)

Epoch 1/10, Loss: 1.703461961673044
Epoch 2/10, Loss: 1.329960721837895
Epoch 3/10, Loss: 1.1849558109517597
Epoch 4/10, Loss: 1.0859901697739311
Epoch 5/10, Loss: 1.0049093243716014
Epoch 6/10, Loss: 0.9402447010550048
Epoch 7/10, Loss: 0.8834659727028263
Epoch 8/10, Loss: 0.8353202411585756
Epoch 9/10, Loss: 0.7933586020298931
Epoch 10/10, Loss: 0.7557180290636809
Test Accuracy: 71.11%


可以预期，最终的方法会比 CosineLoss 效果更好，因为现在我们在教师和学生之间引入了一个可训练层，这为学生的学习提供了一定的灵活空间，而不是强制学生直接复制教师的表示。引入额外的网络正是基于提示的蒸馏（hint-based distillation）的核心思想。


In [22]:
# 打印教师模型的测试准确率（作为性能上界参考）
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")

# 打印仅使用交叉熵训练的学生模型准确率（基线，无蒸馏）
print(f"Student accuracy without teacher: {test_accuracy_light_ce:.2f}%")

# 打印使用输出蒸馏（知识蒸馏，KD）训练的学生模型准确率
# 该方法在交叉熵基础上增加了软标签损失，温度 T 和权重可调
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

# 打印使用隐藏表示余弦损失训练的学生模型准确率
# 该方法在交叉熵基础上增加了余弦嵌入损失，强制学生隐藏向量与教师对齐（通过池化降维）
print(f"Student accuracy with CE + CosineLoss: {test_accuracy_light_ce_and_cosine_loss:.2f}%")

# 打印使用特征图 MSE 损失（基于提示的蒸馏）训练的学生模型准确率
# 该方法在交叉熵基础上增加了 MSE 损失，让学生通过一个额外的可训练回归器学习教师的特征图
print(f"Student accuracy with CE + RegressorMSE: {test_accuracy_light_ce_and_mse_loss:.2f}%")

Teacher accuracy: 74.84%
Student accuracy without teacher: 70.55%
Student accuracy with CE + KD: 70.13%
Student accuracy with CE + CosineLoss: 70.83%
Student accuracy with CE + RegressorMSE: 71.11%


**结论**
==========
上述方法均不会增加网络的参数量或推理时间，因此性能提升仅以训练时计算梯度的微小代价实现。在机器学习应用中，我们主要关注推理时间，因为训练是在模型部署前完成的。如果我们的轻量化模型在部署时仍然过于笨重，可以采用不同的思路，例如训练后量化。额外的损失函数不仅可以用于分类任务，还可应用于多种任务，你可以尝试调整系数、温度或神经元数量等参数。欢迎调整上述教程中的任何数值，但请记住，如果改变神经元/卷积核的数量，很可能会导致形状不匹配。

更多信息请参阅：

Hinton, G., Vinyals, O., Dean, J.: Distilling the knowledge in a neural network. In: Neural Information Processing System Deep Learning Workshop (2015)
Romero, A., Ballas, N., Kahou, S.E., Chassang, A., Gatta, C., Bengio, Y.: Fitnets: Hints for thin deep nets. In: Proceedings of the International Conference on Learning Representations (2015)